In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("tam_eslesme_sonuclari.tsv", sep="\t")

In [3]:
df.columns

Index(['Ingredient', 'IDs', '__index_level_0__', 'I-NAME tokens',
       'Matched Words', 'En Uzun Tam Eşleşme'],
      dtype='object')

In [4]:
df.head()

,Ingredient,IDs,__index_level_0__,I-NAME tokens,Matched Words,En Uzun Tam Eşleşme
0,low-sodium vegetable,"0, 11434, 14768, 82089, 96363, 105999, 194867,...",0,"▁vegeta, ble",vegetable,vegetable
1,chicken stock,"0, 11, 94, 101, 130, 155, 200, 345, 552, 678, ...",1,"▁chicken, ▁stock","chicken, stock",chicken stock
2,dried brown lentils,"0, 17104, 836751, 838386, 841486, 871267, 8744...",2,"▁brown, ▁lenti, ls","brown, lentils",brown lentils
3,dried french green lentils,"0, 2684, 3173, 3530, 16691",3,"▁fren, ch, ▁green, ▁lenti, ls","french, green, lentils",french green lentils
4,"celery, chopped","0, 1149, 2427, 3530, 5081, 5340, 5891, 8907, 9...",4,"▁cele, ry",celery,celery


In [5]:
df.tail()

,Ingredient,IDs,__index_level_0__,I-NAME tokens,Matched Words,En Uzun Tam Eşleşme
549649,s sweet mustard sauce for pretzels and more,1171959,549651,"▁sweet, ▁must, ard, ▁sauce","sweet, mustard, sauce",sweet mustard sauce
549650,orange-ginger broiled pork tenderloin,1172243,549652,"▁orang, e, ging, er, led, ▁por, k, ▁tender, lo...","orange-ginger, broiled, pork, tenderloin",orange-ginger broiled pork tenderloin
549651,garlic herb infused skewers,1173044,549653,"▁gar, lic, ▁her, b","garlic, herb",garlic herb
549652,alfredo noodles mix,1173818,549654,"▁al, fred, o, ▁nood, les, ▁mix","alfredo, noodles, mix",alfredo noodles mix
549653,instant devil's food pudding mix,1173972,549655,"▁instant, ▁de, vil, ', s, ▁food, ▁pu, dding, ▁mix","instant, devil's, food, pudding, mix",instant devil's food pudding mix


In [6]:
def get_unique_id_set(df, match_column="En Uzun Tam Eşleşme", id_column="IDs"):
    df["word_count"] = df[match_column].fillna("").apply(lambda x: len(x.split()))

    filtered_df = df[df["word_count"] > 4]

    unique_id_set = set(filtered_df[id_column].unique())

    return unique_id_set

In [7]:
unique_id_set = get_unique_id_set(df)

In [8]:
len(unique_id_set)

36692

In [9]:
df = pd.read_parquet("merged_and_cleaned_all_data.parquet", engine="pyarrow")

In [10]:
# The recipes with 130350, 151885, 275952, 326407, 381514, 428479, 445989, 450835, 646986, 653053, 717383, 749572, 808681 IDs must be deleted.
ids_to_remove = [130350, 151885, 275952, 326407,381514, 428479, 445989, 450835, 646986, 653053, 717383, 749572, 808681]
index_names = df[df['ID'].isin(ids_to_remove)].index
df.drop(index_names, inplace=True)

In [11]:
unique_id_list = []
for item in unique_id_set:
    parts = item.split(",")  
    for p in parts:
        p = p.strip()  
        unique_id_list.append(int(p)) 


In [12]:
unique_id_list

[173314,
 274224,
 14148,
 676298,
 644572,
 144532,
 671362,
 784523,
 470982,
 5762,
 647238,
 275688,
 242065,
 142733,
 727817,
 803655,
 523367,
 344580,
 476280,
 619455,
 184515,
 475,
 171813,
 412551,
 634980,
 819151,
 781205,
 748187,
 465718,
 254916,
 700328,
 336275,
 613081,
 673443,
 196335,
 219644,
 439691,
 775723,
 544905,
 62215,
 268305,
 515267,
 172453,
 154347,
 791488,
 70173,
 416257,
 456200,
 796794,
 158532,
 346533,
 376861,
 136077,
 530278,
 613067,
 1394,
 586487,
 173368,
 175120,
 108070,
 524859,
 664235,
 73097,
 645798,
 626428,
 241984,
 286394,
 480578,
 791392,
 649722,
 82180,
 101259,
 163961,
 173812,
 199052,
 239206,
 338907,
 392018,
 402837,
 423442,
 444951,
 615418,
 633306,
 639278,
 665539,
 701821,
 732250,
 738399,
 748469,
 808702,
 594587,
 367364,
 395420,
 518756,
 708687,
 713805,
 802410,
 85306,
 912158,
 436529,
 98926,
 348111,
 732392,
 113024,
 340034,
 564336,
 600917,
 792294,
 128473,
 511535,
 385429,
 370999,
 13920

In [13]:
filtered_df = df[df["ID"].isin(unique_id_list)]
print(len(filtered_df))
filtered_df.to_parquet("problematic_recipes.parquet", engine="pyarrow")

102096


In [14]:
remaining_df = df[~df["ID"].isin(unique_id_list)]
remaining_df.to_parquet("non-problematic_recipes.parquet", engine="pyarrow")